In [1]:
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install pennylane pennylane-lightning jupyter jax jaxlib optax scikit-learn scikit-image pybind11

In [3]:
# 1. 패키지 버전 충돌을 일으키는 원인(setuptools, packaging)을 최신으로 업데이트합니다.
%pip install --upgrade pip setuptools packaging

# 2. 문제의 원인이 될 수 있는 기존 lightning을 지웁니다.
%pip uninstall -y pennylane-lightning

# 3. [중요] 일반적인 방법으로 lightning을 설치합니다.
%pip install pennylane-lightning

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.0 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 58.1.0
    Uninstalling setuptools-58.1.0:
      Successfully uninstalled setuptools-58.1.0
Note: you may need to restart the kernel to use updated packages.
Found existing installation: pennylane_lightning 0.42.0
Uninstalling pennylane_lightning-0.42.0:
  Successfully uninstalled pennylane_lightning-0.42.0
Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --------- ------------------------------ 1.6/6.6 MB 8.3 MB/s eta 0:00:01
   ---------------------- ----------------- 3.7/6.6 MB 9.5 MB/s eta 0:00:01
   ----------------------------------- ---- 5.8/6.6 MB 9.5 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 9.2 MB/s  0:00:00
Note: you may need to restart the 

In [4]:
import os
# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"

import pennylane as qml
from pennylane import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

import jax
import jax.numpy as jnp
import optax
import numpy as np

# ----------------------------------------------
# [데이터셋 전처리부] (skimage 없이 순수 numpy 사용)
# ----------------------------------------------
def prepare_quantum_dataset(classes=(3,6), img_size=(4,4), test_size=0.2, random_state=42):
    """
    실제 손글씨 숫자 MNIST 데이터셋을 양자 회로에 넣기 전 전처리하는 함수

    Parameters:
    - classes: 이진 분류할 두 개의 숫자 클래스
    - img_size: 축소 이미지 해상도
    - test_size: 테스트 데이터 분리 비율

    Returns:
    - X_train, X_test: [샘플 수, n_qubits] 형태의 이진 픽셀 배열 (0 또는 1)
    - Y_train, Y_test: [샘플 수] 형태의 레이블 배열 (1 또는 -1)
    """
    # 1. 데이터셋 로드
    digits = load_digits()
    X_raw, Y_raw = digits.data, digits.target
    
    # 2. 지정된 클래스(예시: 3과 6)만 필터링
    mask = np.isin(Y_raw, classes)
    X_filtered, Y_filtered = X_raw[mask], Y_raw[mask]
    
    # 3. 레이블 양자 관측값(Pauli-Z)에 맞게 +1, -1로 변환
    Y_binary = np.where(Y_filtered == classes[0], 1.0, -1.0)

    # 4. 이미지 해상도 축소 (8x8 -> 4x4) 및 픽셀 값 정규화 및 이진화
    n_samples = X_filtered.shape[0]
    X_resized = np.zeros((n_samples, 16))

    for i in range(n_samples):
        # 4.1. 8x8 이미지를 4x4로 축소
        img_8x8 = X_filtered[i].reshape(8, 8)
        img_4x4 = img_8x8.reshape(4, 2, 4, 2).mean(axis=(1, 3))
        img_binary = np.where(img_4x4 > img_4x4.mean(), 1.0, 0.0)
        X_resized[i] = img_binary.flatten()

    # 5. 학습용/검증용 데이터 분리
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_resized, Y_binary, test_size=test_size, random_state=random_state
        , stratify=Y_binary # stratify 옵션을 사용하여 클래스 비율 유지
    )

    # 경사하강법이 미분 가능하도록 requires_grad 설정
    # X_train = np.array(X_train, requires_grad=False)
    # Y_train = np.array(Y_train, requires_grad=False)
    # X_test = np.array(X_test, requires_grad=False)
    # Y_test = np.array(Y_test, requires_grad=False)

    print("[데이터셋 정보]")
    print(f"이미지 해상도: {img_size[0]}x{img_size[1]}, 입력 큐비트 수: {n_qubits}")
    print(f"학습용 샘플 수: {X_train.shape[0]}, 검증용 샘플 수: {X_test.shape[0]}")

    return X_train, X_test, Y_train, Y_test

# ----------------------------------------------
# [퀀텀 컴퓨팅 부] JAX 인터페이스 적용
# ----------------------------------------------
n_qubits = 17
dev = qml.device("default.qubit", wires=n_qubits) # 느릴 경우 "lightning.qubit"로 변경 가능 (C++ backend)

@qml.qnode(dev, interface="jax", diff_method="backprop") # lightning.qubit + jax 인터페이스 사용
def quantum_neural_net(inputs, weights):
    """
    양자 회로 정의
    Parameters:
    - inputs: 입력 데이터 (샘플 수, n_qubits)
    - weights: 양자 회로의 파라미터 (샘플 수, n_qubits) - theta

    Returns:
    - qml.expval(qml.PauliZ(0)): 판독 큐비트의 측정 결과 반환
    """
    # 16개의 입력
    for i in range(16):
        qml.RX(inputs[i] * jnp.pi, wires=i)
    
    for i in range(n_qubits):
        qml.RY(weights[i], wires=i)

    for i in range(n_qubits-1):
        qml.CNOT(wires=[i, i+1])
    
    # 3. 측정 결과를 고전 컴퓨터로 전송
    # 내부적으로 확률적 관측 수행하여 기댓값 도출
    return qml.expval(qml.PauliZ(0))  # 첫 번째 큐비트의 측정 결과 반환

# [퀀텀 컴퓨팅 부] 배치 처리를 위한 vmap(벡터화) 정의
    # 이제 batched_qnn은 한 개의 데이터가 아닌 280개(Full Batch) 행렬 전체를 한 방에 통과시킵니다.
    # in_axes=(0, None) -> 데이터(inputs)는 차원을 쪼개서 넣고, weights는 그대로 넣으라는 뜻
batched_qnn = jax.vmap(quantum_neural_net, in_axes=(0, None))

# ----------------------------------------------
#                [고전 컴퓨팅 부]
# ----------------------------------------------
def loss_function(weights, X, Y):
    """
    손실 함수 정의 (Mean Squared Error)
    """
    predictions = batched_qnn(X, weights)
    return jnp.mean((predictions - Y) ** 2)

loss_and_grad_fn = jax.jit(jax.value_and_grad(loss_function))


# ---------------------------------------------------------
# [실제 수행] 데이터 초기화, 손실 함수, 업데이트 루프
# ---------------------------------------------------------

# 1. 데이터 로드 후 JAX 배열(DeviceArray)로 변환
X_train, X_test, Y_train, Y_test = prepare_quantum_dataset()
X_train_jax = jnp.array(X_train)
Y_train_jax = jnp.array(Y_train)
X_test_jax = jnp.array(X_test)

# 2. 초기 세타 값 설정 (JAX의 난수 생성기 사용)
key = jax.random.PRNGKey(42)
theta = jax.random.normal(key, (n_qubits,))

# 3. Optax 최적화기 설정 (JAX 전용 경사하강법 라이브러리)
learning_rate = 0.1
optimizer = optax.sgd(learning_rate)
opt_state = optimizer.init(theta)

epochs = 300

print("\n🚀 Full-Batch JAX 학습 시작...")
for epoch in range(epochs):
    # 단 한 줄로 전체 데이터에 대한 손실값과 미분값을 구함! (초고속)
    current_loss, grads = loss_and_grad_fn(theta, X_train_jax, Y_train_jax)
    
    # 파라미터 업데이트
    updates, opt_state = optimizer.update(grads, opt_state)
    theta = optax.apply_updates(theta, updates)
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch: {epoch+1:3d} | Loss: {current_loss:.4f}")

print("\n학습 완료!")

# 테스트 검증 (테스트 역시 VMAP으로 한 방에 처리)
Y_test_pred = batched_qnn(X_test_jax, theta)
predicted_labels = jnp.where(Y_test_pred > 0, 1, -1)
accuracy = jnp.mean(predicted_labels == jnp.array(Y_test))
print(f"테스트 정확도: {accuracy * 100:.2f}%")

[데이터셋 정보]
이미지 해상도: 4x4, 입력 큐비트 수: 17
학습용 샘플 수: 291, 검증용 샘플 수: 73

🚀 Full-Batch JAX 학습 시작...
Epoch:  10 | Loss: 1.0406
Epoch:  20 | Loss: 1.0005
Epoch:  30 | Loss: 1.0000
Epoch:  40 | Loss: 1.0000
Epoch:  50 | Loss: 1.0000
Epoch:  60 | Loss: 1.0000
Epoch:  70 | Loss: 1.0000
Epoch:  80 | Loss: 1.0000
Epoch:  90 | Loss: 1.0000


XlaRuntimeError: FAILED_PRECONDITION: Buffer Definition Event: Error preparing computation: %sOut of memory allocating 10984883424 bytes.